In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [2]:
IMG_SIZE = 128
BATCH_SIZE = 16
NUM_CLASSES = 7

In [5]:
#Load Dataset

train_df = pd.read_csv("Datasets/train.csv")
valid_df = pd.read_csv("Datasets/validation.csv")
test_df = pd.read_csv("Datasets/test.csv")

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)
metadata = pd.read_csv("Datasets/HAM10000_metadata.csv")

metadata.head()

(7010, 11)
(1502, 11)
(1503, 11)


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [6]:
image_dir1 = "Datasets/HAM10000_images_part_1"
image_dir2 = "Datasets/HAM10000_images_part_2"

image_path = {}

for folder in [image_dir1, image_dir2]:

    for file in os.listdir(folder):

        image_id = file.split(".")[0]

        image_path[image_id] = os.path.join(folder, file)

metadata["path"] = metadata["image_id"].map(image_path)

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0027419.jpg
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025030.jpg
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0026769.jpg
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025661.jpg
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Datasets/HAM10000_images_part_2/ISIC_0031633.jpg


In [7]:
encoder = LabelEncoder()

metadata["label"] = encoder.fit_transform(metadata["dx"])

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0027419.jpg,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025030.jpg,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0026769.jpg,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025661.jpg,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Datasets/HAM10000_images_part_2/ISIC_0031633.jpg,2


In [8]:
train_df, temp_df = train_test_split(

    metadata,

    test_size=0.30,

    stratify=metadata["label"],

    random_state=42
)

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label"],

    random_state=42
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(7010, 9)
(1502, 9)
(1503, 9)


In [9]:
train_df["label"] = train_df["label"].astype(str)
val_df["label"] = val_df["label"].astype(str)
test_df["label"] = test_df["label"].astype(str)

In [10]:
train_df["path"] = train_df["path"].astype(str)
val_df["path"] = val_df["path"].astype(str)
test_df["path"] = test_df["path"].astype(str)

In [11]:
classes = np.unique(train_df["label"])

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=train_df["label"]
)

class_weights = dict(zip(range(len(classes)), weights))

print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


In [12]:
from tensorflow.keras.applications.efficientnet import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8,1.2]
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# NOW create the generators
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


# Early Stop

In [13]:
#DenseNet121
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense

base_model = DenseNet121(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)

)

base_model.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

densenet_model = Model(inputs, outputs)

densenet_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [14]:
#Compile
densenet_model.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [15]:
early_stop = EarlyStopping(

    monitor="val_accuracy",

    mode="max",

    patience=3,

    restore_best_weights=True,

    verbose=1

)

In [18]:
#Train
history_dense = densenet_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,
    callbacks=[early_stop],

    class_weight=class_weights)



Epoch 1/5


I0000 00:00:1785404458.726698  394409 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


439/439 ━━━━━━━━━━━━━━━━━━━━ 798s 2s/step - accuracy: 0.4488 - loss: 1.6499 - val_accuracy: 0.5386 - val_loss: 1.3658
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 773s 2s/step - accuracy: 0.5583 - loss: 1.2887 - val_accuracy: 0.6278 - val_loss: 1.1067
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 718s 2s/step - accuracy: 0.5884 - loss: 1.1835 - val_accuracy: 0.6372 - val_loss: 1.0771
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 572s 1s/step - accuracy: 0.5973 - loss: 1.1251 - val_accuracy: 0.5666 - val_loss: 1.2136
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 468s 1s/step - accuracy: 0.6147 - loss: 1.0755 - val_accuracy: 0.6072 - val_loss: 1.1185
Restoring model weights from the end of the best epoch: 3.


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = densenet_model.evaluate(train_generator, verbose=0)

val_loss, val_acc = densenet_model.evaluate(val_generator, verbose=0)

test_loss, test_acc = densenet_model.evaluate(test_generator, verbose=0)

pred = densenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [20]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.6148359775543213 0.6371504664421082 0.6074517369270325 0.7338001530254468 0.6074517631403858 0.652256607427207


In [22]:
densenet_model.save("models2/densenet_model_early.keras")

# lr

In [23]:
#DenseNet121
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense

base_model = DenseNet121(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)

)

base_model.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

densenet_model = Model(inputs, outputs)

densenet_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [24]:
reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=2,

    min_lr=1e-6,

    verbose=1

)

In [25]:
densenet_model.compile(

    optimizer=Adam(learning_rate=0.0005),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [26]:
history_lr = densenet_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    callbacks=[reduce_lr]

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 391s 865ms/step - accuracy: 0.4763 - loss: 1.4834 - val_accuracy: 0.6352 - val_loss: 0.9850 - learning_rate: 5.0000e-04
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 731s 2s/step - accuracy: 0.5890 - loss: 1.1928 - val_accuracy: 0.5846 - val_loss: 1.0992 - learning_rate: 5.0000e-04
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6051 - loss: 1.0885
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
439/439 ━━━━━━━━━━━━━━━━━━━━ 716s 2s/step - accuracy: 0.6051 - loss: 1.0885 - val_accuracy: 0.5925 - val_loss: 1.1740 - learning_rate: 5.0000e-04
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 613s 1s/step - accuracy: 0.6481 - loss: 0.9651 - val_accuracy: 0.6418 - val_loss: 0.9496 - learning_rate: 2.5000e-04
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 724s 2s/step - accuracy: 0.6486 - loss: 0.9277 - val_accuracy: 0.6165 - val_loss: 1.0258 - learning_rate: 2.5000e-04


In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = densenet_model.evaluate(train_generator, verbose=0)

val_loss, val_acc = densenet_model.evaluate(val_generator, verbose=0)

test_loss, test_acc = densenet_model.evaluate(test_generator, verbose=0)

pred = densenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [28]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.6523537635803223 0.616511344909668 0.6074517369270325 0.7545309287320071 0.6074517631403858 0.6538081159308767


In [30]:
densenet_model.save("models2/densenet_model_Lr.keras")

# sgd

In [31]:
#DenseNet121
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense

base_model = DenseNet121(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)

)

base_model.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

densenet_model = Model(inputs, outputs)

densenet_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [32]:
from tensorflow.keras.optimizers import SGD

densenet_model.compile(

    optimizer=SGD(

        learning_rate=0.0001,

        momentum=0.9,

        nesterov=True

    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [33]:
history_sgd = densenet_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 405s 898ms/step - accuracy: 0.2262 - loss: 1.7787 - val_accuracy: 0.3862 - val_loss: 1.6708
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 659s 2s/step - accuracy: 0.4536 - loss: 1.4634 - val_accuracy: 0.4507 - val_loss: 1.4865
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 677s 1s/step - accuracy: 0.5090 - loss: 1.3682 - val_accuracy: 0.5406 - val_loss: 1.3144
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 705s 2s/step - accuracy: 0.5464 - loss: 1.2911 - val_accuracy: 0.5739 - val_loss: 1.2357
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 755s 2s/step - accuracy: 0.5672 - loss: 1.2407 - val_accuracy: 0.5832 - val_loss: 1.2047


In [34]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = densenet_model.evaluate(train_generator, verbose=0)

val_loss, val_acc = densenet_model.evaluate(val_generator, verbose=0)

test_loss, test_acc = densenet_model.evaluate(test_generator, verbose=0)

pred = densenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [35]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.5934379696846008 0.5832223892211914 0.5781769752502441 0.7243324160034392 0.5781769793745841 0.6225064212464808


In [36]:
densenet_model.save("models2/densenet_model_SGD.keras")

# RMS

In [37]:
#DenseNet121
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense

base_model = DenseNet121(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)

)

base_model.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

densenet_model = Model(inputs, outputs)

densenet_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [38]:
from tensorflow.keras.optimizers import RMSprop

densenet_model.compile(

    optimizer=RMSprop(

        learning_rate=0.001

    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [39]:
history_dense = densenet_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 753s 2s/step - accuracy: 0.5136 - loss: 1.6078 - val_accuracy: 0.5619 - val_loss: 1.2708
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 707s 2s/step - accuracy: 0.5692 - loss: 1.3603 - val_accuracy: 0.5173 - val_loss: 1.3659
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 690s 1s/step - accuracy: 0.5909 - loss: 1.2761 - val_accuracy: 0.5479 - val_loss: 1.4918
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 633s 1s/step - accuracy: 0.6061 - loss: 1.2504 - val_accuracy: 0.6651 - val_loss: 0.9450
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 597s 1s/step - accuracy: 0.6031 - loss: 1.2859 - val_accuracy: 0.6831 - val_loss: 0.9314


In [40]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = densenet_model.evaluate(train_generator, verbose=0)

val_loss, val_acc = densenet_model.evaluate(val_generator, verbose=0)

test_loss, test_acc = densenet_model.evaluate(test_generator, verbose=0)

pred = densenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [41]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.7084165215492249 0.6830891966819763 0.6573519706726074 0.7399267911501 0.6573519627411843 0.6814656788089722


In [42]:
densenet_model.save("models2/densenet_model_rms.keras")

# Batch

In [43]:
train_generator_32 = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=True

)

val_generator_32 = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

test_generator_32 = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [44]:
#DenseNet121
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense

base_model = DenseNet121(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)

)

base_model.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

densenet_model = Model(inputs, outputs)

densenet_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_9 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [45]:
densenet_model.compile(

    optimizer=Adam(learning_rate=0.001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [46]:
history_batch32 = densenet_model.fit(

    train_generator_32,

    validation_data=val_generator_32,

    epochs=5,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 606s 3s/step - accuracy: 0.4770 - loss: 1.5305 - val_accuracy: 0.5979 - val_loss: 1.1279
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 545s 2s/step - accuracy: 0.5893 - loss: 1.2003 - val_accuracy: 0.6305 - val_loss: 1.0342
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 578s 3s/step - accuracy: 0.6068 - loss: 1.1144 - val_accuracy: 0.6758 - val_loss: 0.9018
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6218 - loss: 1.0188

KeyboardInterrupt: 

In [48]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = densenet_model.evaluate(train_generator_32, verbose=0)

val_loss, val_acc = densenet_model.evaluate(val_generator_32, verbose=0)

test_loss, test_acc = densenet_model.evaluate(test_generator_32, verbose=0)

pred = densenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [49]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.6145506501197815 0.6125166416168213 0.5974717140197754 0.7463171156795321 0.5974717232202262 0.6410666310327199


In [50]:
densenet_model.save("models2/densenet_model_batch.keras")

# hypertuning

In [16]:
from keras_tuner import HyperModel
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.optimizers import Adam, RMSprop

In [17]:
class DenseNetHyperModel(HyperModel):

    def build(self, hp):

        base_model = DenseNet121(

            weights="imagenet",

            include_top=False,

            input_shape=(IMG_SIZE, IMG_SIZE, 3)

        )

        base_model.trainable = False

        model = Sequential([

            base_model,

            GlobalAveragePooling2D(),

            Dense(

                hp.Choice(
                    "dense_units",
                    [128,256]
                ),

                activation="relu"

            ),

            Dropout(

                hp.Choice(
                    "dropout_rate",
                    [0.3,0.5]
                )

            ),

            Dense(
                NUM_CLASSES,
                activation="softmax"
            )

        ])

        optimizer = hp.Choice(

            "optimizer",

            ["adam","rmsprop"]

        )

        learning_rate = hp.Choice(

            "learning_rate",

            [1e-3,1e-4]

        )

        if optimizer == "adam":

            opt = Adam(
                learning_rate=learning_rate
            )

        else:

            opt = RMSprop(
                learning_rate=learning_rate
            )

        model.compile(

            optimizer=opt,

            loss="categorical_crossentropy",

            metrics=["accuracy"]

        )

        return model

In [18]:
from keras_tuner import RandomSearch

tuner = RandomSearch(

    DenseNetHyperModel(),

    objective="val_accuracy",

    max_trials=3,

    executions_per_trial=1,

    directory="DenseNet_tuning",

    project_name="DenseNet"

)

In [19]:
tuner.search(

    train_generator,

    validation_data=val_generator,

    epochs=3

)

Trial 3 Complete [00h 03m 38s]
val_accuracy: 0.6877496838569641

Best val_accuracy So Far: 0.7077230215072632
Total elapsed time: 00h 15m 35s


In [20]:
best_hp = tuner.get_best_hyperparameters(1)[0]

print(best_hp.values)

{'dense_units': 256, 'dropout_rate': 0.3, 'optimizer': 'adam', 'learning_rate': 0.001}


In [21]:
best_model = tuner.hypermodel.build(best_hp)

history = best_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 78s 162ms/step - accuracy: 0.2987 - loss: 2.9366 - val_accuracy: 0.3995 - val_loss: 1.6536
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 71s 162ms/step - accuracy: 0.3408 - loss: 1.7573 - val_accuracy: 0.4807 - val_loss: 1.4406
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 69s 158ms/step - accuracy: 0.3111 - loss: 1.7306 - val_accuracy: 0.3162 - val_loss: 1.7092
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 69s 158ms/step - accuracy: 0.3715 - loss: 1.6544 - val_accuracy: 0.3868 - val_loss: 1.5923
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 69s 158ms/step - accuracy: 0.3912 - loss: 1.6511 - val_accuracy: 0.4647 - val_loss: 1.4030


In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = densenet_model.evaluate(train_generator, verbose=0)

val_loss, val_acc = densenet_model.evaluate(val_generator, verbose=0)

test_loss, test_acc = densenet_model.evaluate(test_generator, verbose=0)

pred = densenet_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [23]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.2049928605556488 0.20639148354530334 0.20958083868026733 0.40993861786371855 0.20958083832335328 0.27211387721760116


In [25]:
densenet_model.save("hyper_dense.keras")

# Fine Tuning

In [31]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

IMG_SIZE = (224, 224)
NUM_CLASSES = len(train_generator.class_indices)

# Load pretrained DenseNet121
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(128, 128, 3)
)

# Freeze all layers
base_model.trainable = False

# Unfreeze only the last 10 layers
# Unfreeze only the last 5 layers
for layer in base_model.layers[-5:]:
    layer.trainable = True

# Train for fewer epochs


optimizer = Adam(learning_rate=5e-6)

# Build classifier
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
   epochs = 5,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 78s 162ms/step - accuracy: 0.5248 - loss: 1.5028 - val_accuracy: 0.6591 - val_loss: 1.1929 - learning_rate: 1.0000e-05
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 69s 157ms/step - accuracy: 0.6181 - loss: 1.2948 - val_accuracy: 0.6644 - val_loss: 1.1308 - learning_rate: 1.0000e-05
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 69s 157ms/step - accuracy: 0.6270 - loss: 1.2431 - val_accuracy: 0.6638 - val_loss: 1.0959 - learning_rate: 1.0000e-05
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 69s 157ms/step - accuracy: 0.6258 - loss: 1.2099 - val_accuracy: 0.6671 - val_loss: 1.0728 - learning_rate: 1.0000e-05
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 70s 160ms/step - accuracy: 0.6287 - loss: 1.1996 - val_accuracy: 0.6684 - val_loss: 1.0573 - learning_rate: 1.0000e-05


In [32]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = model.evaluate(train_generator, verbose=0)

val_loss, val_acc = model.evaluate(val_generator, verbose=0)

test_loss, test_acc = model.evaluate(test_generator, verbose=0)

pred = model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [33]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.6624821424484253 0.6684420704841614 0.6679973602294922 0.5196231304998845 0.6679973386560213 0.5673977697185945


In [34]:
densenet_model.save("TUNe_dense.keras")

In [35]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Model": [
        "Early Stopping",
        "Learning Rate Scheduler",
        "SGD",
        "RMSprop",
        "Batch-wise Comparison",
        "Hyperparameter Tuning",
        "Fine-Tuning"
    ],
    "Train Accuracy": [
        0.6148359775543213,
        0.6523537635803223,
        0.5934379696846008,
        0.7084165215492249,
        0.6145506501197815,
        0.2049928605556488,
        0.6624821424484253
    ],
    "Validation Accuracy": [
        0.6371504664421082,
        0.616511344909668,
        0.5832223892211914,
        0.6830891966819763,
        0.6125166416168213,
        0.20639148354530334,
        0.6684420704841614
    ],
    "Test Accuracy": [
        0.6074517369270325,
        0.6074517369270325,
        0.5781769752502441,
        0.6573519706726074,
        0.5974717140197754,
        0.20958083868026733,
        0.6679973602294922
    ],
    "Precision": [
        0.7338001530254468,
        0.7545309287320071,
        0.7243324160034392,
        0.7399267911501,
        0.7463171156795321,
        0.40993861786371855,
        0.5196231304998845
    ],
    "Recall": [
        0.6074517631403858,
        0.6074517631403858,
        0.5781769793745841,
        0.6573519627411843,
        0.5974717232202262,
        0.20958083832335328,
        0.6679973386560213
    ],
    "F1-Score": [
        0.652256607427207,
        0.6538081159308767,
        0.6225064212464808,
        0.6814656788089722,
        0.6410666310327199,
        0.27211387721760116,
        0.5673977697185945
    ]
})

# Sort by Test Accuracy (Highest to Lowest)
comparison_df = (
    comparison_df.sort_values("Test Accuracy", ascending=False)
                 .reset_index(drop=True)
                 .round(4)
)

comparison_df

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Precision,Recall,F1-Score
0,Fine-Tuning,0.6625,0.6684,0.6680,0.5196,0.6680,0.5674
1,RMSprop,0.7084,0.6831,0.6574,0.7399,0.6574,0.6815
2,Early Stopping,0.6148,0.6372,0.6075,0.7338,0.6075,0.6523
3,Learning Rate Scheduler,0.6524,0.6165,0.6075,0.7545,0.6075,0.6538
4,Batch-wise Comparison,0.6146,0.6125,0.5975,0.7463,0.5975,0.6411
5,SGD,0.5934,0.5832,0.5782,0.7243,0.5782,0.6225
6,Hyperparameter Tuning,0.2050,0.2064,0.2096,0.4099,0.2096,0.2721
